## Great, now that we discussed a little let's continue

Given that the current approach utilized by the authors lacks reproducibility, we will explore an alternative method by leveraging nf-core pipelines for data analysis.

Please explain, how we will achieve reproducibility for the course  with this approach.


> By using the nf-core pipeline, we get a consistent output format&consistent outputs due to nf-core pipelines reproduceability standards. This also makes further documentation not that relevant, since nf-core contains all the documentation for the pipeline and it's enough to explain the choosen parameters instead of the whole workflow.

You have successfully downloaded 2 of the fastq files we will use in our study.

What is the next step if we want to first have a count table and check the quality of our fastq files? What is the pipeline called to do so?

> To achieve this, first we need to do quality control (QC) followed by a mapping of the transcript to a reference . A pipeline that includes those steps is nf-core/rnaseq [https://nf-co.re/rnaseq/3.27.0], thus I will use this for the following steps.  

Analyze the 2 files using an nf-core pipeline.

What does this pipeline do?

Which are the main tools that will be used in the pipeline?

> The pipeline works in 5 steps:
1. Pre-processing: Adapter-trimming, Quality control: TrimGalore, FastQC
2. Genome alignment & quantification (so a gene count matrix): HISAT2 & Salmon
3. Pseudo-alignment & quantification: Salmon
4. Post-processing (Deduplication): Picard
5. Quality control & reporting: MultiQC

As all other nf-core pipelines, the chosen pipeline takes in a samplesheet as input.

Use Python and pandas to create the samplesheet for your 2 samples. Feel free to make use of the table you created earlier today.

Choose your sample names wisely, they must be the connection of the results to the metadata. If you can't find the sample in the metadata later, the analysis was useless.

In [4]:
# This code creates a samplesheet for rnaseq from the one fetchngs produces
import pandas as pd

samplesheet = pd.read_csv("./fetchngs_out/samplesheet/samplesheet.csv")
samplesheet = samplesheet.rename(columns={"instrument_platform": "seq_platform"}) # Different name for this col required by rnaseq
samplesheet["strandedness"] = "auto" # set to auto, since we dont know it better lol
samplesheet.to_csv("./samplesheet_rnaseq.csv", index=False)

In [ ]:
# post here the command you used to run nf-core/rnaseq
!nextflow run nf-core/rnaseq \
    -r 3.27.0 \
    -profile docker \
    --input ./samplesheet_rnaseq.csv \
    -params-file ./rnaseq_params.yaml \
    --outdir ./rnaseq_out \
    -c ./nextflow.config

Explain all the parameters you set and why you set them in this way.



```bash
# post here the command you used to run nf-core/rnaseq
!nextflow run nf-core/rnaseq \
    -r 3.27.0 \                             # version used
    -profile docker \                       # docker as container plattform
    --input ./samplesheet_rnaseq.csv \      # samplesheet created earlier to provide the fastq files
    -params-file ./rnaseq_params.yaml \     # further parameters set in seperate file, params set there are discussed below
    --outdir ./rnaseq_out \                 # Folder for output
    -c ./nextflow.config                    # I need to specify this since resources are limited on my laptop

```

Further parameters:
- fasta: "https://ftp.ensembl.org/pub/release-102/fasta/mus_musculus/dna/Mus_musculus.GRCm38.dna.primary_assembly.fa.gz"
> Link to refernce genome I found myself, hope this is ok
- gtf: "https://ftp.ensembl.org/pub/release-102/gtf/mus_musculus/Mus_musculus.GRCm38.102.gtf.gz"
> same for gtf (file that specifies gene loci on genome)
- skip_alignment: true
- skip_pseudo_alignment: true
> I needed to skip align&pseudoalign, since otherwise the workflow wasn't viable on my laptop. 

## Browsing the results

How did the pipeline perform?

> My run for only 2 samples and without alignment/pseudo-alignment already took 35m 24s, so for the full pipeline for all samples, it would take way longer than this. This would make the full analysis absolutely unfeasible for my hardware; server usage would be necessary. Info about the job can be found in the `pipeline_info/execution_report*` file.

Explain the quality control steps. Are you happy with the quality and why. If not, why not.
Please give additional information on : 
- ribosomal rRNA
- Duplication
- GC content

What are the possible steps that could lead to poorer results?

- If hight rRNA is observed, this could be due to an failed depletion or enrichment step during library-prep.
- Duplication could be due to PCR over-amplification leading to many unwanted duplicates. 
- Different GC contents could be due to contamination <br>
=> all affect the quality of the downstream analysis and are thus not wanted in our data.

Would you exclude any samples? If yes, which and why?

I would definetly exclude `SRX19144492` since it has a GC-content of 63%, which is higher than normal for most of mammals.

Further I would exclude `SRX19144481` due to it's high rRNA containment:

![](./images/high_rRNA.png)

Lastly I would also exclude `SRX19144489` due to its extreme duplication rate of ~88%, especially since other runs have low duplication rates <30% such as `SRX19144485`

What would you now do to continue the experiment? What are the scientists trying to figure out? Which packages on R or python would you use?
> Now the main goal of the paper could be followed: find genes/pathways that are differentially expressed across the conditions and are thus (probably) linked to opioid addiction/withdrawl. For this a pipeline like `nf-core/differentialabundance` or a custom python script using scanpy with anndata could be used.

Look at differential gene expression to see, how genotypes/conditions affect the gene regulation.